# Group A Full Run
This notebook runs the complete Group A SDXL Base 1.0 + IP-Adapter workflow for HW13. It is intended for execution in the Colab Web UI on an H100.

What this notebook does:
- Mounts Google Drive and resolves the project root
- Installs required packages
- Loads the shared Group A pipeline
- Executes the full 149-image Group A workflow with checkpointing
- Runs the Step 3 verification checks
- Packages results into a downloadable ZIP with a logs folder

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
%pip install -q diffusers transformers accelerate safetensors genaibook jiwer scipy pillow matplotlib

In [ ]:
from pathlib import Path
import os
import sys

DRIVE_ROOT = Path('/content/drive/MyDrive')
CANDIDATE_ROOTS = [
    DRIVE_ROOT / 'kamp_hw13',
    DRIVE_ROOT / 'Visual_Studio__UC_Spring_26' / 'GEN_AI' / 'HW1' / 'kamp_hw13',
]

def find_project_root():
    checked_paths = []
    for candidate in CANDIDATE_ROOTS:
        checked_paths.append(candidate)
        if (candidate / 'hw13_scripts' / 'kamp_hw13.py').exists():
            return candidate

    checked_display = '\n'.join(f' - {path}' for path in checked_paths)
    raise FileNotFoundError(
        'Could not locate kamp_hw13 on Google Drive.\n'
        'Upload the project to /content/drive/MyDrive/kamp_hw13\n'
        'or mirror the local nesting under /content/drive/MyDrive/Visual_Studio__UC_Spring_26/GEN_AI/HW1/kamp_hw13.\n'
        f'Checked:\n{checked_display}'
    )

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / 'hw13_scripts'))

print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'Module exists: {(PROJECT_ROOT / "hw13_scripts" / "kamp_hw13.py").exists()}')

In [ ]:
from kamp_hw13 import load_group_a_pipeline, run_group_a
from hw13_data_utils import TeeLogger, cleanup_pipeline

FULL_RESULTS_ROOT = PROJECT_ROOT / 'hw13_experiments'
FULL_LOG_PATH = PROJECT_ROOT / 'hw13_printouts' / 'group_a_full_log.txt'

print(f'FULL_RESULTS_ROOT = {FULL_RESULTS_ROOT}')
print(f'FULL_LOG_PATH = {FULL_LOG_PATH}')

In [ ]:
pipeline = None
tee = TeeLogger(FULL_LOG_PATH)
tee_started = False

try:
    pipeline, device = load_group_a_pipeline()
    tee.start()
    tee_started = True
    print(f'[NOTEBOOK] Loaded Group A full pipeline on {device}')
    summary = run_group_a(
        dry_run=False,
        pipeline=pipeline,
        experiments_dir=FULL_RESULTS_ROOT,
        checkpoint_enabled=True,
        resume=True,
    )
    print(summary)
finally:
    if tee_started:
        tee.stop()
    if pipeline is not None:
        cleanup_pipeline(pipeline, 'Group A full pipeline')

In [ ]:
from pathlib import Path

EXP_DIR = PROJECT_ROOT / 'hw13_experiments'
NB_DIR = PROJECT_ROOT / 'hw13_scripts' / 'notebooks'

assert (NB_DIR / 'group_a_dry_run.ipynb').exists(), 'FAIL: Dry-run notebook missing'
assert (NB_DIR / 'group_a_full.ipynb').exists(), 'FAIL: Full notebook missing'
print('TEST 0 PASS: Both Group A notebooks exist.')

ga18b_base = list((EXP_DIR / 'exp1_baselines').glob('ga18b_*.png'))
ga18c_base = list((EXP_DIR / 'exp1_baselines').glob('ga18c_*.png'))
assert len(ga18b_base) >= 1, f'FAIL: Expected ≥1 GA18B baseline, found {len(ga18b_base)}'
assert len(ga18c_base) >= 1, f'FAIL: Expected ≥1 GA18C baseline, found {len(ga18c_base)}'
print(f'TEST 1 PASS: {len(ga18b_base)} GA18B + {len(ga18c_base)} GA18C baseline image(s).')

exp2_files = list((EXP_DIR / 'exp2_ga18b_scale').glob('ga18b_*.png'))
assert len(exp2_files) >= 45, f'FAIL: Expected ≥45 Exp 2 images, found {len(exp2_files)}'
print(f'TEST 2 PASS: {len(exp2_files)} Exp 2 images.')

exp3_files = list((EXP_DIR / 'exp3_ga18c_blocks').glob('ga18c_*.png'))
assert len(exp3_files) >= 40, f'FAIL: Expected ≥40 Exp 3 images, found {len(exp3_files)}'
print(f'TEST 3 PASS: {len(exp3_files)} Exp 3 images.')

exp4_files = list((EXP_DIR / 'exp4_text_image_interaction').glob('ga18*.png'))
assert len(exp4_files) >= 28, f'FAIL: Expected ≥28 Exp 4 images, found {len(exp4_files)}'
print(f'TEST 4 PASS: {len(exp4_files)} Exp 4 images.')

seed_ga18b = list((EXP_DIR / 'exp7_seed_sensitivity').glob('ga18b_*.png'))
seed_ga18c = list((EXP_DIR / 'exp7_seed_sensitivity').glob('ga18c_*.png'))
assert len(seed_ga18b) >= 8, f'FAIL: Expected ≥8 GA18B seed images, found {len(seed_ga18b)}'
assert len(seed_ga18c) >= 8, f'FAIL: Expected ≥8 GA18C seed images, found {len(seed_ga18c)}'
print(f'TEST 5 PASS: {len(seed_ga18b)} GA18B + {len(seed_ga18c)} GA18C seed sensitivity images.')

for exp_name, min_rows in [('exp2_ga18b_scale', 45), ('exp3_ga18c_blocks', 40), ('exp4_text_image_interaction', 28)]:
    csv_path = EXP_DIR / exp_name / f'{exp_name}.csv'
    assert csv_path.exists(), f'FAIL: CSV missing: {csv_path}'
    with csv_path.open() as handle:
        rows = sum(1 for _ in handle) - 1
    assert rows >= min_rows, f'FAIL: {exp_name} CSV has {rows} rows, expected ≥{min_rows}'
print('TEST 6 PASS: All Group A CSVs exist with correct row counts.')

all_images = exp2_files + exp3_files + exp4_files + seed_ga18b + seed_ga18c
for img in all_images:
    assert img.stat().st_size > 1000, f'FAIL: Suspiciously small image: {img} ({img.stat().st_size} bytes)'
print(f'TEST 7 PASS: All {len(all_images)} images are non-empty.')

print('\n✅ STEP 3 VERIFICATION: ALL TESTS PASSED')

In [ ]:
from datetime import datetime
import shutil

bundle_root = PROJECT_ROOT / 'hw13_experiments' / '_group_a_bundle'
if bundle_root.exists():
    shutil.rmtree(bundle_root)

logs_dir = bundle_root / 'logs'
logs_dir.mkdir(parents=True, exist_ok=True)

paths_to_copy = [
    PROJECT_ROOT / 'hw13_experiments' / 'exp1_baselines',
    PROJECT_ROOT / 'hw13_experiments' / 'exp2_ga18b_scale',
    PROJECT_ROOT / 'hw13_experiments' / 'exp3_ga18c_blocks',
    PROJECT_ROOT / 'hw13_experiments' / 'exp4_text_image_interaction',
    PROJECT_ROOT / 'hw13_experiments' / 'exp7_seed_sensitivity',
]

for src in paths_to_copy:
    if src.exists():
        shutil.copytree(src, bundle_root / src.name, dirs_exist_ok=True)

for log_name in ['group_a_full_log.txt', 'kamp_hw13_execution_log.txt']:
    log_path = PROJECT_ROOT / 'hw13_printouts' / log_name
    if log_path.exists():
        shutil.copy2(log_path, logs_dir / log_path.name)

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
archive_base = PROJECT_ROOT / 'hw13_experiments' / f'group_a_results_{timestamp}'
archive_path = shutil.make_archive(str(archive_base), 'zip', root_dir=bundle_root)
print(f'Created ZIP: {archive_path}')

In [ ]:
try:
    from google.colab import files
    files.download(archive_path)
except Exception as error:
    print(f'Download step skipped: {error}')
    print(f'Manually download: {archive_path}')